# Llama-3.1-8B-Instruct QLoRA Fine-Tune on umarbutler/better-cuad

This notebook fine-tunes a legal QA model using the CUAD-derived dataset on Hugging Face.
Target: `meta-llama/Meta-Llama-3.1-8B-Instruct` with QLoRA.

**GPU requirements:** A100 (40GB) recommended. T4 (15GB) is possible but slow (~15-20h for 3 epochs).
**Note:** Llama 3.1 is a gated model. You must:
1. Accept the licence at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
2. Set your HF_TOKEN below.


In [ ]:
# Install dependencies
!pip -q install -U transformers datasets peft bitsandbytes accelerate

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
import os

# --- REQUIRED: set your Hugging Face token (Llama 3.1 is gated) ---
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

# --- CHANGED: model and output dir ---
model_name = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
output_dir = 'llama3.1-8b-legal-qlora'
max_len = 1024
num_train_epochs = 3
train_limit = None  # set to int (e.g. 2000) for a quick test run


In [ ]:
# Reuse fine-tuned weights if available
from pathlib import Path
reuse_choice = input("Reuse existing fine-tuned weights if found? [y/N]: ").strip().lower()
reuse_if_available = reuse_choice in {"y", "yes"}
output_path = Path(output_dir)
weight_files = [
    "adapter_model.safetensors", "adapter_model.bin", "adapter_config.json",
    "pytorch_model.bin", "model.safetensors"
]
weights_exist = any((output_path / f).exists() for f in weight_files)
SKIP_TRAINING = bool(reuse_if_available and weights_exist)
if SKIP_TRAINING:
    print(f"Found weights in {output_dir}. Training will be skipped.")
elif reuse_if_available:
    print("No saved weights found. Training will run.")
else:
    print("Training will run.")


In [ ]:
# --- CHANGED: bfloat16 throughout (Llama 3.1 was trained in bf16) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,   # was float16
    bnb_4bit_quant_type="nf4"
)

# --- CHANGED: added token= and padding_side ---
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True,
    trust_remote_code=True,
    token=os.environ.get('HF_TOKEN')
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for causal LM training

# --- CHANGED: added torch_dtype=bfloat16 and token= ---
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    token=os.environ.get('HF_TOKEN')
)

if SKIP_TRAINING:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, output_dir)
else:
    # --- CHANGED: use_gradient_checkpointing=True ---
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True
    )
    # --- CHANGED: r=16, target_modules already correct for Llama ---
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()


In [ ]:
# Build QA prompts using the tokenizer's chat template (works for any instruct model)
if SKIP_TRAINING:
    print("Skipping dataset prep (reusing weights).")
else:
    from datasets import load_dataset

    ds = load_dataset("umarbutler/better-cuad")
    raw = ds["train"]

    if train_limit:
        raw = raw.select(range(min(train_limit, len(raw))))

    answer_cols = [c for c in raw.column_names if c.endswith("-Answer")]
    text_col = "Text"
    max_questions_per_doc = 8

    # --- CHANGED: use chat template instead of raw string prompt ---
    def build_batch(batch):
        texts = []
        for i in range(len(batch[text_col])):
            text = batch[text_col][i]
            if not text:
                continue
            count = 0
            for col in answer_cols:
                ans = batch[col][i]
                if ans is None:
                    continue
                ans = str(ans).strip()
                if not ans or ans.lower() in {"n/a", "na", "none", "no"}:
                    continue
                clause = col[:-7]  # remove "-Answer"
                messages = [
                    {
                        "role": "system",
                        "content": (
                            "You are a legal contract assistant. "
                            "Answer using only the provided context. "
                            "Always cite the specific clause your answer is based on. "
                            "If the answer cannot be found in the contract, say so explicitly."
                        )
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Highlight the parts (if any) of this contract related to {clause}.\n\n"
                            f"Context:\n{text}"
                        )
                    },
                    {
                        "role": "assistant",
                        "content": ans
                    }
                ]
                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=False
                )
                texts.append(prompt)
                count += 1
                if count >= max_questions_per_doc:
                    break
        return {"text": texts}

    qa = raw.map(build_batch, batched=True, remove_columns=raw.column_names)
    qa = qa.shuffle(seed=42)
    split = qa.train_test_split(test_size=0.05, seed=42)
    train = split["train"]
    eval_ = split["test"]

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_len)

    train_tok = train.map(tokenize, batched=True, remove_columns=["text"])
    eval_tok  = eval_.map(tokenize, batched=True, remove_columns=["text"])


In [ ]:
# Train
if SKIP_TRAINING:
    print(f"Skipping training. Using weights from {output_dir}")
else:
    from transformers import TrainingArguments

    train_args_kwargs = dict(
        output_dir=output_dir,
        # --- CHANGED: batch_size 2->1 (8B memory), accumulation 8->16 (keeps effective batch ~same) ---
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        # --- CHANGED: lr 2e-4->1e-4 (larger models prefer lower LR) ---
        learning_rate=1e-4,
        num_train_epochs=num_train_epochs,
        # --- CHANGED: bf16 instead of fp16 ---
        bf16=True,
        fp16=False,
        # --- CHANGED: gradient checkpointing + paged adamw (essential for fitting 8B) ---
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        max_grad_norm=0.3,
        logging_steps=50,
        save_steps=100,
        save_total_limit=2,
        report_to="none",
    )

    # Handle eval_strategy vs evaluation_strategy across transformers versions
    if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        train_args_kwargs["eval_strategy"] = "steps"
        train_args_kwargs["eval_steps"] = 100
    else:
        train_args_kwargs["evaluation_strategy"] = "steps"
        train_args_kwargs["eval_steps"] = 100

    args = TrainingArguments(**train_args_kwargs)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        data_collator=data_collator,
    )

    trainer.train()
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)


In [ ]:
!pip -q install pypdf faiss-cpu

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = next(iter(uploaded))

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += page.extract_text() or ""
print("Chars:", len(text))


In [ ]:
def chunk_text(t, size=1200, overlap=200):
    t = " ".join(t.split())
    chunks = []
    i = 0
    while i < len(t):
        chunks.append(t[i:i+size])
        i += size - overlap
    return chunks

chunks = chunk_text(text)
print("Chunks:", len(chunks))


In [ ]:
import numpy as np
import torch
import faiss
from transformers import AutoTokenizer as EmbTok, AutoModel

emb_model = "BAAI/bge-base-en-v1.5"
emb_tok = EmbTok.from_pretrained(emb_model)
emb_net = AutoModel.from_pretrained(emb_model).eval()

def embed(texts):
    inputs = emb_tok(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        out = emb_net(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)
        pooled = (out * mask).sum(dim=1) / mask.sum(dim=1)
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
    return pooled.cpu().numpy().astype("float32")

embeddings = embed(chunks)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

def retrieve(query, k=4):
    q = embed([query])
    scores, idx = index.search(q, k)
    return [chunks[i] for i in idx[0]]


In [ ]:
import re

NAME_LINE_RE = re.compile(r"Name:\s*([^\n\r]+)")
PARTY_QUERY_RE = re.compile(
    r"\b(who|name|names|identify|list)\b.*\b(contract[\s-]?holders?|tenant|landlord|parties?)\b"
    r"|\b(contract[\s-]?holders?|tenant|landlord|parties?)\b.*\b(who|name|names|identify|list)\b",
    re.IGNORECASE,
)

def extract_names_from_text(text):
    lines = NAME_LINE_RE.findall(text)
    return [l.strip() for l in lines if l.strip()]

# --- CHANGED: use chat template for inference, increased max_new_tokens 128->256 ---
def answer_from_pdf(question, k=4):
    if PARTY_QUERY_RE.search(question):
        names = extract_names_from_text(text)
        return "Names found: " + "; ".join(names)

    ctx = "\n\n".join(retrieve(question, k))

    messages = [
        {
            "role": "system",
            "content": (
                "You are a legal contract assistant. "
                "Answer using only the provided context. "
                "Cite the specific clause or section your answer is based on. "
                "If the answer cannot be found in the contract, say so explicitly."
            )
        },
        {
            "role": "user",
            "content": f"Question: {question}\n\nContext:\n{ctx}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


In [ ]:
# Example usage:
# print(answer_from_pdf("What are the termination rights and notice periods?"))
# print(answer_from_pdf("Who are the contract-holders?"))
# print(answer_from_pdf("What are the payment obligations?"))
